Kawasaki Quantum Summer Camp 2026

# 量子化学シミュレーション：サンプルベースの量子対角化

Kifumi Numata, IBM Quantum (Aug 03, 2026)

必要なライブラリーをインストールします。

In [ ]:
%pip install 'qiskit[visualization]' qiskit-ibm-runtime ffsim qiskit-addon-sqd

このノートブックでは、**サンプルベースの量子対角化（SQD）** を使い、実機の量子コンピューター上で窒素分子（N₂）の **基底状態エネルギー** を推定します。

SQD は、量子・古典のハイブリッドアルゴリズムです。量子コンピューターがパラメーター化された回路を実行して、どの電子配置が最も重要かを *提案* し、一方で古典コンピューターが線形代数を行います。つまり、小さく賢く選ばれた配置の集合の中で分子ハミルトニアンを対角化し、固有エネルギーを求めます。

# 0. 量子化学の基礎

このノートブックを完了するのに化学の背景知識は **必要ありません**。この節では、必要最小限の語彙だけを紹介します。

### 分子、軌道、電子

分子を小さなコンサートホールと考えてください。
**分子軌道** はそのホールの座席です。
**電子** は、1つの厳格な規則に従って着席しなければならない客です。すなわち、各座席（軌道）は *お客であるスピンを高々1人* しか収容できません（パウリの排他原理と呼ばれます）。

各軌道は2つの電子を収容できます。1つは **スピン α（↑）**、もう1つは **スピン β（↓）** です。
私たちは常に2つの種類のスピンを別々に追跡します。

### 基底状態

**基底状態** は、全エネルギーが最も低い座り方となる配置です。
基底状態を知ることは化学者に、分子が安定かどうか、その結合がどれだけ強いか、そして反応するかどうかを教えてくれます。
基底状態エネルギーを正確に計算することは、量子コンピューターの近い将来の最も魅力的な応用の1つです。というのも、可能となる座り方の数が *指数的* に増えるからです。$n$ 個の軌道と $N_e$ 個の電子については、スピンの種類ごとに ${}_nC_{N_e}$ 通りの配置があり、この計算はどんな古典コンピューターの手にもすぐに負えなくなります。
このノートブックの N₂ では、活性軌道 16 個と1スピン種につき電子 5 個なので、すでに1スピン種あたり ${}_{16}C_{5} = 4{,}368$ 通りの配置があります。

基底状態を見つけることは、形式的には **固有値問題** です。

$$\hat{H} |\psi_0\rangle = E_0 |\psi_0\rangle$$

ここで $\hat{H}$は分子のハミルトニアン 、$|\psi_0\rangle$ は基底状態の波動関数、$E_0$ は最も低いエネルギー固有値です。

分子のハミルトニアンとは、エネルギーに相当する演算子で、電子や原子核の運動、電子と原子核の引き合うエネルギー、電子同士が反発するエネルギー全てをまとめて、分子全体のエネルギーを計算するための数式です。
座席がスピンで埋まっているかどうかで表現したハミルトニアンは第二量子化と呼ばれ、以下の式で表されます。スピンのあるなしが量子ビットの状態 1, 0 と自然な対応が可能になります。

$$\hat{H} = \sum_{pq} h_{pq}\,\hat{a}^\dagger_p \hat{a}_q + \frac{1}{2}\sum_{pqrs} g_{pqrs}\,\hat{a}^\dagger_p \hat{a}^\dagger_q \hat{a}_s \hat{a}_r + E_\text{nuc}$$

- $h_{pq}$: **ー電子積分** - 電子の運動エネルギーと電子・原子核間の引力（コード中の `hcore`）。
- $g_{pqrs}$: **二電子反発積分** - 電子同士の反発力（コード中の `eri`）。
- $\hat{a}^\dagger_p$: **生成演算子** - 電子を1個軌道 $p$ に置く演算。
- $\hat{a}_p$: **消滅演算子** - 電子を1個軌道 $p$ から取り除く演算。
- $E_\text{nuc}$: 原子核間の反発エネルギー。

SQD はこの問題を、$\hat{H}$ を小さな **部分空間**に射影し、その部分空間の中で厳密に対角化することで解きます。
量子コンピューターの仕事は、どの状態ベクトルがそこに属するかを提案することです。

### 古典的な手法

2つの古典的な基底エネルギーを計算する手法が、私たちの計算の参照的な枠組みになります。

| 手法 | 何をするか | 精度 |
|---|---|---|
| **ハートリー・フォック（HF）** | 各電子が相関なしに、独立して最も安い座席を選ぶ。エネルギーの低い席から順番に埋まった状態。計算は速いが、電子同士の反発を無視する。 | $E_0$ の上限。粗い見積もり。 |
| **CCSD**（Coupled Cluster Singles & Doubles） | HF から始め、代数的な振幅 $t_1$（シングル）と $t_2$（ダブル）を使って、HF 配置からの1・2電子の「ジャンプ」を加える。相関のほとんどを捉える。 | $E_0$ の効率的で妥当な古典的見積もりを与える。 |

> **重要な点:** CCSD は量子回路の種として *使う* こともあります。その振幅は電子が軌道間をどうジャンプしたがるかを記述しており、私たちはその化学的な直観を量子アンザッツ(仮の回路)のパラメーターに直接移植します。

# 1. N₂ 分子とそのハミルトニアンを構築する

### 窒素分子

**分子状窒素（N₂）** を扱います。1.0 Å 離れた2つの窒素原子です。

### 活性空間と凍結軌道

すべての軌道が同じように重要というわけではありません。
最も内側の **コア** 軌道はエネルギー的に深いところにあり、本質的に決して変化しません。分子が何をしようと、それらはそこにいて完全に満足しています。
それらの軌道を量子力学的に計算するのは、貴重な量子ビットの無駄になります。

そこで、それらを **凍結** します。
- `n_frozen = 2` は、エネルギーが最も低い2つのコア軌道を量子計算から外し凍結します。
- `active_space = range(n_frozen, mol.nao_nr())` は、化学的に興味深いこと（結合の形成、電子の相関）が実際に起こる軌道が集合した活性空間です。

これが **活性空間近似** です。まず全体について完全な HF を解き、その後、活性空間の電子だけを量子コンピューター上で扱います。
凍結の後、残るのは次のとおりです。
- `num_orbitals` 個の活性空間軌道 → これがスピン種あたりの量子ビット数に等しくなります。
- スピン α とスピン β の `(num_elec_a, num_elec_b)` 個の活性電子。

下のコードセルはまた、**一電子積分**（`hcore`）、**二電子反発積分**（`eri`）、そして一定の **原子核間の反発エネルギー** を取り出します。これらを合わせたものが、活性空間に制限されたハミルトニアン $\hat{H}$ の数学的表現であり、後で古典的な固有値ソルバーが対角化を行うものです。

`pyscf` という量子化学計算をPythonで行うためのライブラリーを使って準備していきます。

In [ ]:
import pyscf
from pyscf import mcscf

# N2分子を構築する
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, 0)], ["N", (1.0, 0, 0)]],  
    basis="6-31g",
)

# アクティブスペースを定義する
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# 分子積分を計算する
scf = pyscf.scf.RHF(mol).run()    # HF近似でのエネルギー計算
num_orbitals = len(active_space)    # 活性空間の軌道の数
n_electrons = int(sum(scf.mo_occ[active_space]))    # スピン種あたりの量子ビット数
num_elec_a = (n_electrons + mol.spin) // 2    # スピン α の活性電子数
num_elec_b = (n_electrons - mol.spin) // 2    # スピン β の活性電子数
cas = pyscf.mcscf.CASCI(scf, num_orbitals, (num_elec_a, num_elec_b)) # 計算を行うための活性空間を設定
mo = cas.sort_mo(active_space, base=0)    # 活性空間軌道の指定
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)    # 一電子積分と原子核間の反発エネルギー
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), num_orbitals)    # 二電子反発積分

### 量子アンザッツ(仮の回路)の種にするために CCSD を実行する

量子回路を構築する前に、**CCSD** を古典的に実行します。
CCSD を最終的な答えとして使うのではなく、その **振幅** を量子回路の作成に使います。

- `t1[i, a]` — 1つの電子が占有軌道 $i$ から仮想軌道 $a$ へ「ジャンプ」する振幅。
- `t2[i, j, a, b]` — 電子 *対* が $(i, j)$ から $(a, b)$ へ散乱する振幅。

これらの振幅は、電子がどう相関するかについての CCSD の化学的な直観を符号化しています。
次の節で、量子化学用回路作成ライブラリー `ffsim` がそれらを読み取って量子回路の回転角にコンパイルし、古典的な化学の知識を高品質な初期推定として量子デバイスに移植します。

In [ ]:
# アンザッツを初期化するために CCSD の t2 振幅を取得する
import pyscf
import pyscf.cc
ccsd = pyscf.cc.CCSD(scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]).run(max_cycle=1000)
t1 = ccsd.t1
t2 = ccsd.t2

# 2. 量子状態の準備

状態準備の目標は、N₂ の基底状態を十分よく近似する量子回路を構築し、それをサンプリングすると真の基底状態に近い電子配置が得られるようにすることです。
今回は、**LUCJ アンザッツ** — **Local Unitary Cluster Jastrow** — と呼ばれる、ハードウェア効率の良いパラメーター化された回路を使います。

### CCSD 振幅から回路パラメーターへ

`ffsim.UCJOpSpinBalanced.from_t_amplitudes(t1, t2, n_reps, interaction_pairs)` は、CCSD の1電子・2電子励起振幅を読み取り、LUCJ アンザッツの回転角 と 結合係数 にコンパイルします。
直観的には、CCSD が「軌道 $i$ と $a$ が強く入れ替わる」と言えば、対応する回転角度が大きくなります。

### 相互作用ペア — ハードウェアの制約

`interaction_pairs` 引数は、どの軌道のペアが相互作用できるかを設定します。

- `alpha_alpha_indices = [(p, p+1) ...]` — 量子ビット鎖に沿った同一スピンの *最近接* 相互作用（チップ上でスピン種ごとに1本の線）。下の図で赤と青の鎖。
- `alpha_beta_indices` — **スピン種をまたぐ** 相互作用: 位置 $p$ の α 量子ビットが位置 $q$ の β 量子ビットと結合する。
  これらは **デバイスの配線によって制約されます**: 2つの物理量子ビットがチップ上で隣接している場合にのみ、その結合はハードウェアネイティブです。下の図で紫の部分。

<img src="https://quantum.cloud.ibm.com/assets-docs-learning/_next/image?url=%2Fdocs%2Fimages%2Ftutorials%2Fimproving-energy-estimation-of-a-fermionic-hamiltonian-with-sqd%2F7e0ee7e1-2d24-417f-ac59-25c58db79aa9.avif&w=1920&q=75" alt="lucj_ansatz" width="400">

In [ ]:
n_reps = 1
alpha_alpha_indices = [(p, p + 1) for p in range(num_orbitals - 1)]
alpha_beta_indices = [(p, p) for p in range(0, num_orbitals, 4)]

import ffsim
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=n_reps,
    interaction_pairs=(alpha_alpha_indices, alpha_beta_indices),
)

nelec = (num_elec_a, num_elec_b)

# 空の量子回路を作成
from qiskit import QuantumCircuit, QuantumRegister
qubits = QuantumRegister(2 * num_orbitals, name="q")
circuit = QuantumCircuit(qubits)

# Hartree-Fock状態を基準状態として準備し、それを量子回路に追加
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(num_orbitals, nelec), qubits)

# 基準状態にUCJ演算子を適用
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
circuit.measure_all()

circuit.decompose().decompose().draw("mpl", fold =-1)

# 3. 実験を実行する

Colabの左側の鍵マーク「シークレット」をクリックし、「IBM_QUANTUM_API」と「IBM_QUANTUM_CRN」のノートブックからのアクセスをオンにします。

![image.png](https://github.com/quantum-tokyo/kawasaki-quantum-camp/blob/main/day3/secrets.jpg?raw=true)

In [ ]:
from google.colab import userdata
from qiskit_ibm_runtime import QiskitRuntimeService

api_key = userdata.get("IBM_QUANTUM_API")
crn     = userdata.get("IBM_QUANTUM_CRN")

service = QiskitRuntimeService(
    channel="ibm_cloud",
    token=api_key,
    instance=crn,
)
service.backends()

量子コンピューターのデバイスを指定します。

In [ ]:
# 以下で使うデバイスを指定できます。
backend = service.backend('ibm_fez') 

In [ ]:
#一番空いているデバイスを自動的に選択することもできます
backend = service.least_busy(operational=True)
print("最も空いているバックエンドは: ", backend)

デバイス上の物理量子ビットを指定し、回路を実機で実行できるようにトランスパイルします。

In [ ]:
spin_a_layout = [13, 12, 11, 18, 31, 30, 29, 38, 49, 48, 47, 57, 67, 66, 65, 77]
spin_b_layout = [15, 19, 35, 34, 33, 39, 53, 52, 51, 58, 71, 70, 69, 78, 89, 88]
initial_layout = spin_a_layout + spin_b_layout

from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, initial_layout=initial_layout
)

# このパスマネージャーによって生成された回路をハードウェア実行に使用します。
pass_manager.pre_init = ffsim.qiskit.PRE_INIT
isa_circuit = pass_manager.run(circuit)
print(f"Gate counts (w/ pre-init passes): {isa_circuit.count_ops()}")

回路を実行します。

In [ ]:
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime import SamplerOptions
opts = SamplerOptions()
opts.dynamical_decoupling.enable = True
opts.twirling.enable_measure = True

sampler = Sampler(mode=backend, options=opts)
job = sampler.run([isa_circuit], shots=100_000)
my_job_id = job.job_id()
print("job id:", my_job_id)
job.status()

jobの状況を確認します。

In [ ]:
job.status()

実行待ちに時間がかかるので、過去に計算した結果のデータをロードします。

In [ ]:
import numpy as np
import urllib.request, io
url = "https://raw.githubusercontent.com/quantum-tokyo/kawasaki-quantum-camp/main/day3/N2_device_bitarray_hr.npy"

with urllib.request.urlopen(url) as r:
    bit_array = np.load(io.BytesIO(r.read()), allow_pickle=True).item()

# 4. サンプルベースの量子対角化 (SQD) を使用した結果の後処理

このセクションでは、量子回路から生成・測定された結果を後処理して、分子の基底状態エネルギーを推定します。データサンプルを修正することを繰り返し、精度を向上させます。

ここで、ユーザーが設定できるオプションがいくつかあります。

* `max_iterations`: ループの反復回数。
* `num_batches`: 取り出すバッチの数 
* `samples_per_batch`: 各バッチに含めるサンプル数
* `max_cycles`: 固有状態ソルバーの最大サイクル数


In [ ]:
%%time
from qiskit_addon_sqd.fermion import SCIResult, diagonalize_fermionic_hamiltonian
import numpy as np

# SQDのオプション
energy_tol = 1e-3  
occupancies_tol = 1e-3 
max_iterations = 3

# 固有状態ソルバーのオプション
num_batches = 5
samples_per_batch = 50
symmetrize_spin = True 
carryover_threshold = 1e-4 
max_cycle = 200
rng = np.random.default_rng(24)

# 中間結果を保存するためのリスト
result_history = [] 

def callback(results: list[SCIResult]): 
    result_history.append(results)
    iteration = len(result_history)
    print(f"Iteration {iteration}")
    for i, result in enumerate(results):
        print(f"\tSubsample {i}")
        print(f"\t\tEnergy: {result.energy + nuclear_repulsion_energy}")
        print(f"\t\tSubspace dimension: {np.prod(result.sci_state.amplitudes.shape)}")

result = diagonalize_fermionic_hamiltonian(
    hcore, # 一電子積分
    eri, # 二電子積分
    bit_array,
    samples_per_batch=samples_per_batch, # 各バッチに含めるサンプル数
    norb=num_orbitals, # 軌道の数
    nelec=nelec, # スピンの数
    num_batches=num_batches, # 取り出すバッチの数 
    energy_tol=energy_tol, # エネルギーの精度
    occupancies_tol=occupancies_tol, # 占有率の精度
    max_iterations=max_iterations, # ループの反復回数
    carryover_threshold=carryover_threshold, # ループの際の重み
    callback=callback,
    seed=rng,
)

### 結果を表示する

In [ ]:
def plot_energy_and_occupancy(result_history, exact_energy):

    # Data for energies plot
    
    x1 = range(len(result_history))
    min_e = [
        min(result, key=lambda res: res.energy).energy + nuclear_repulsion_energy
        for result in result_history
    ]
    e_diff = [abs(e - exact_energy) for e in min_e]
    yt1 = [1.0, 1e-1, 1e-2, 1e-3, 1e-4]
    
    # Chemical accuracy (+/- 1 milli-Hartree)
    chem_accuracy = 0.001
    
    # Data for avg spatial orbital occupancy
    y2 = np.sum(result.orbital_occupancies, axis=0)
    x2 = range(len(y2))
    
    fig, axs = plt.subplots(1, 2, figsize=(12, 6))
    
    # Plot energies
    axs[0].plot(x1, e_diff, label="energy error", marker="o")
    axs[0].set_xticks(x1)
    axs[0].set_xticklabels(x1)
    axs[0].set_yticks(yt1)
    axs[0].set_yticklabels(yt1)
    axs[0].set_yscale("log")
    axs[0].set_ylim(1e-4)
    axs[0].axhline(y=chem_accuracy, color="#BF5700", linestyle="--", label="chemical accuracy")
    axs[0].set_title("Approximated Ground State Energy Error vs SQD Iterations")
    axs[0].set_xlabel("Iteration Index", fontdict={"fontsize": 12})
    axs[0].set_ylabel("Energy Error (Ha)", fontdict={"fontsize": 12})
    axs[0].legend()
    
    # Plot orbital occupancy
    axs[1].bar(x2, y2, width=0.8)
    axs[1].set_xticks(x2)
    axs[1].set_xticklabels(x2)
    axs[1].set_title("Avg Occupancy per Spatial Orbital")
    axs[1].set_xlabel("Orbital Index", fontdict={"fontsize": 12})
    axs[1].set_ylabel("Avg Occupancy", fontdict={"fontsize": 12})
    
    print(f"Exact energy: {exact_energy:.5f} Ha")
    print(f"SQD energy: {min_e[-1]:.5f} Ha")
    print(f"Absolute error: {e_diff[-1]:.5f} Ha")
    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
exact_energy = -109.10288938
plot_energy_and_occupancy(result_history, exact_energy)

最初のプロットは、得られたエネルギーの近似値が化学的精度からどの程度離れているかを示しています。（一般的には ``1 kcal/mol`` $\approx$ ``1.6 mH``).

2 番目のプロットは、最終の反復後の各空間軌道の平均占有率を示しています。私たちの解では、電子が二つずつ、最初の 5 つの軌道を高い確率で占有していることがわかります。

# 5. 演習 

エネルギー推定の精度を向上させるという課題に取り組みます。先ほどの設定で使ったいくつかのパラメーターを変化させることでパフォーマンスにどのように影響するかを探ります。

<a id="exercise_1"></a>
<div class="alert alert-block alert-success">
    
<b>基底状態エネルギー推定の改良</b> 

チュートリアルの例では、小さなハミルトニアンを使用し、1回の反復で、バッチ数を 5 にして、分子の基底状態エネルギーの初期近似値を取得しました。ただし、精度を向上させる余地は大きくあります。この演習のタスクは、次の2つの主要なパラメーターを試して、エネルギー推定値を向上させることです。

- **サンプリングサイズ** (`samples_per_batch`): バッチあたりのサンプル数を調整して、ハミルトニアンのサイズを定義します。この値を大きくすると精度が向上しますが、計算負荷が増加する可能性があります。
- **バッチ数** (`num_batches`): バッチ数を変更して、各反復での計算効率と結果の精度のバランスを見つけます。

これらのパラメーターを戦略的に使用して、推定される基底状態エネルギーを実際の値にできるだけ近づけることを目指します。ノイズやハードウェア制約の影響を考慮しながら、さまざまな組み合わせをテストしてエラーを最小限に抑えます。

真の基底状態エネルギーにどれだけ近づけるか見てみましょう。
</div>

まず、上記と同じループを実行し、``num_batches`` および ``samples_per_batch`` 引数を入力として受け取る関数を定義します。

In [ ]:
def sqd_configuration_recovery(num_batches: int, samples_per_batch: int) -> float:
    # SQDのオプション
    energy_tol = 1e-3  
    occupancies_tol = 1e-3 
    max_iterations = 3
    
    # 固有状態ソルバーのオプション
    symmetrize_spin = True 
    carryover_threshold = 1e-4 
    max_cycle = 200
    rng = np.random.default_rng(24)
    
    result = diagonalize_fermionic_hamiltonian(
        hcore,
        eri,
        bit_array,
        samples_per_batch=samples_per_batch,
        norb=num_orbitals,
        nelec=nelec,
        num_batches=num_batches,
        energy_tol=energy_tol,
        occupancies_tol=occupancies_tol,
        max_iterations=max_iterations,
        symmetrize_spin=symmetrize_spin,
        carryover_threshold=carryover_threshold,
        callback=callback,
        seed=rng,
    )
    return result_history

In [ ]:
### この下にコードを記入してください ###

num_batches = 
samples_per_batch = 

### この行以降のコードは変更しないでください ###

In [ ]:
%%time
result_history = [] 
energy_hist = sqd_configuration_recovery(num_batches, samples_per_batch)

In [ ]:
plot_energy_and_occupancy(result_history, exact_energy)

あなたの演習の結果は、4章の結果より良くなりましたか？

# 6. あなたの実機実行結果の確認

そろそろ、実機での実行が終わっているかもしれません。jobの状況を確認します。

In [ ]:
job = service.job(my_job_id) 
job.status()

上記のセルを何回か実行して、'DONE' が表示されたら、実機での実行が終わっているので、以下のセルを実行して結果を確認します。

In [ ]:
### 'DONE'になってから実行します ###
primitive_result = job.result()
pub_result = primitive_result[0]
bit_array = pub_result.data.meas

In [ ]:
# 0次元の object 配列に入れて pickle 保存する
arr = np.empty((), dtype=object)
arr[()] = bit_array
np.save("N2_device_bitarray_hr.npy", arr, allow_pickle=True)

In [ ]:
%%time
result_history = [] 
energy_hist = sqd_configuration_recovery(num_batches, samples_per_batch)

In [ ]:
plot_energy_and_occupancy(result_history, exact_energy)

### References
- Boseong Kim, ["QGSS2026 Lab4c"](https://github.com/qiskit-community/qgss-2026/blob/main/lab-4/QGSS2026_Lab4c_ja.ipynb)